# 00 - Data Exploration

Exploratory analysis of the CheXpert-v1.0-small dataset.
Examines class distributions, label types (positive, negative, uncertain),
pathology co-occurrence patterns, and sample chest X-ray images.

## 1. Load and Inspect the Dataset

Import the required libraries, define the project paths and target pathologies, extract the dataset, and load the training, validation, and test metadata files.


In [1]:
import os
import zipfile
import shutil
import subprocess
import matplotlib

import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

from PIL import Image
from matplotlib.patches import Rectangle

matplotlib.rcParams['font.family'] = 'Times New Roman'
matplotlib.rcParams['font.size'] = 11
matplotlib.rcParams['axes.labelsize'] = 12
matplotlib.rcParams['axes.titlesize'] = 13
matplotlib.rcParams['savefig.dpi'] = 600
matplotlib.rcParams['savefig.bbox'] = 'tight'

In [ ]:
DATA_DIR = os.path.join('..', 'data', 'raw')
FIGURES_DIR = os.path.join('..', 'results', 'figures')

os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(FIGURES_DIR, exist_ok=True)

TARGET_PATHOLOGIES = [
    'Atelectasis',
    'Cardiomegaly',
    'Consolidation',
    'Edema',
    'Pleural Effusion'
]

In [3]:
zip_path = os.path.join(DATA_DIR, 'chexpert.zip')

print(f"Extracting {zip_path} ...")
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(DATA_DIR)

print("Extraction complete.")

Extracting ..\data\raw\chexpert.zip ...
Extraction complete.


In [4]:
train_df = pd.read_csv(os.path.join(DATA_DIR, 'train.csv'))
test_df = pd.read_csv(os.path.join(DATA_DIR, 'groundtruth.csv'))
val_df = pd.read_csv(os.path.join(DATA_DIR, 'valid.csv'))

train_df.shape, val_df.shape

((223414, 19), (234, 19))

In [5]:
train_df.columns, val_df.columns

(Index(['Path', 'Sex', 'Age', 'Frontal/Lateral', 'AP/PA', 'No Finding',
        'Enlarged Cardiomediastinum', 'Cardiomegaly', 'Lung Opacity',
        'Lung Lesion', 'Edema', 'Consolidation', 'Pneumonia', 'Atelectasis',
        'Pneumothorax', 'Pleural Effusion', 'Pleural Other', 'Fracture',
        'Support Devices'],
       dtype='str'),
 Index(['Path', 'Sex', 'Age', 'Frontal/Lateral', 'AP/PA', 'No Finding',
        'Enlarged Cardiomediastinum', 'Cardiomegaly', 'Lung Opacity',
        'Lung Lesion', 'Edema', 'Consolidation', 'Pneumonia', 'Atelectasis',
        'Pneumothorax', 'Pleural Effusion', 'Pleural Other', 'Fracture',
        'Support Devices'],
       dtype='str'))

## 2. Inspect Dataset Structure and Label Values

Inspect the data types, columns, and sample rows in the training, validation, and test metadata.

CheXpert labels may take the following values:

- **1.0** — positive finding
- **0.0** — negative finding
- **−1.0** — uncertain finding
- **NaN** — no label recorded

For the later experiments, uncertain labels are handled using the **U-Ones** mapping, where uncertain labels are mapped to **1.0**.


In [6]:
train_df.dtypes

Path                              str
Sex                               str
Age                             int64
Frontal/Lateral                   str
AP/PA                             str
No Finding                    float64
Enlarged Cardiomediastinum    float64
Cardiomegaly                  float64
Lung Opacity                  float64
Lung Lesion                   float64
Edema                         float64
Consolidation                 float64
Pneumonia                     float64
Atelectasis                   float64
Pneumothorax                  float64
Pleural Effusion              float64
Pleural Other                 float64
Fracture                      float64
Support Devices               float64
dtype: object

In [7]:
val_df.dtypes

Path                              str
Sex                               str
Age                             int64
Frontal/Lateral                   str
AP/PA                             str
No Finding                    float64
Enlarged Cardiomediastinum    float64
Cardiomegaly                  float64
Lung Opacity                  float64
Lung Lesion                   float64
Edema                         float64
Consolidation                 float64
Pneumonia                     float64
Atelectasis                   float64
Pneumothorax                  float64
Pleural Effusion              float64
Pleural Other                 float64
Fracture                      float64
Support Devices               float64
dtype: object

In [8]:
test_df.shape, test_df.columns, test_df.dtypes

((500, 15),
 Index(['Study', 'No Finding', 'Enlarged Cardiomediastinum', 'Cardiomegaly',
        'Lung Opacity', 'Lung Lesion', 'Edema', 'Consolidation', 'Pneumonia',
        'Atelectasis', 'Pneumothorax', 'Pleural Effusion', 'Pleural Other',
        'Fracture', 'Support Devices'],
       dtype='str'),
 Study                           str
 No Finding                    int64
 Enlarged Cardiomediastinum    int64
 Cardiomegaly                  int64
 Lung Opacity                  int64
 Lung Lesion                   int64
 Edema                         int64
 Consolidation                 int64
 Pneumonia                     int64
 Atelectasis                   int64
 Pneumothorax                  int64
 Pleural Effusion              int64
 Pleural Other                 int64
 Fracture                      int64
 Support Devices               int64
 dtype: object)

## 3. Inspect Paths and Dataset Splits

Inspect the path and study identifiers used by the training, validation, and test datasets before standardising them for local file access.


In [9]:
train_df.head()

,Path,Sex,Age,Frontal/Lateral,AP/PA,No Finding,Enlarged Cardiomediastinum,Cardiomegaly,Lung Opacity,Lung Lesion,Edema,Consolidation,Pneumonia,Atelectasis,Pneumothorax,Pleural Effusion,Pleural Other,Fracture,Support Devices
0,CheXpert-v1.0-small/train/patient00001/study1/...,Female,68,Frontal,AP,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN,1.0
1,CheXpert-v1.0-small/train/patient00002/study2/...,Female,87,Frontal,AP,NaN,NaN,-1.0,1.0,NaN,-1.0,-1.0,NaN,-1.0,NaN,-1.0,NaN,1.0,NaN
2,CheXpert-v1.0-small/train/patient00002/study1/...,Female,83,Frontal,AP,NaN,NaN,NaN,1.0,NaN,NaN,-1.0,NaN,NaN,NaN,NaN,NaN,1.0,NaN
3,CheXpert-v1.0-small/train/patient00002/study1/...,Female,83,Lateral,NaN,NaN,NaN,NaN,1.0,NaN,NaN,-1.0,NaN,NaN,NaN,NaN,NaN,1.0,NaN
4,CheXpert-v1.0-small/train/patient00003/study1/...,Male,41,Frontal,AP,NaN,NaN,NaN,NaN,NaN,1.0,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN


In [10]:
val_df.head()

,Path,Sex,Age,Frontal/Lateral,AP/PA,No Finding,Enlarged Cardiomediastinum,Cardiomegaly,Lung Opacity,Lung Lesion,Edema,Consolidation,Pneumonia,Atelectasis,Pneumothorax,Pleural Effusion,Pleural Other,Fracture,Support Devices
0,CheXpert-v1.0-small/valid/patient64541/study1/...,Male,73,Frontal,AP,0.0,1.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,CheXpert-v1.0-small/valid/patient64542/study1/...,Male,70,Frontal,PA,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
2,CheXpert-v1.0-small/valid/patient64542/study1/...,Male,70,Lateral,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
3,CheXpert-v1.0-small/valid/patient64543/study1/...,Male,85,Frontal,AP,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,CheXpert-v1.0-small/valid/patient64544/study1/...,Female,42,Frontal,AP,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


## 4. Standardise Image Paths and Verify File Access

Remove the dataset-specific path prefixes from the metadata and verify that the corresponding image files can be located within the local project directory.


In [11]:
test_df.head()

,Study,No Finding,Enlarged Cardiomediastinum,Cardiomegaly,Lung Opacity,Lung Lesion,Edema,Consolidation,Pneumonia,Atelectasis,Pneumothorax,Pleural Effusion,Pleural Other,Fracture,Support Devices
0,CheXpert-v1.0/test/patient64741/study1,0,1,1,1,0,0,0,0,0,0,0,0,0,1
1,CheXpert-v1.0/test/patient64742/study1,0,1,0,1,0,0,0,0,1,0,0,0,0,0
2,CheXpert-v1.0/test/patient64743/study1,0,1,1,1,0,1,0,0,1,0,0,0,0,0
3,CheXpert-v1.0/test/patient64744/study1,0,0,0,0,0,0,0,0,0,0,0,0,0,1
4,CheXpert-v1.0/test/patient64745/study1,0,1,1,1,1,0,0,0,1,0,1,0,0,1


In [12]:
train_df["Path"].head()

0    CheXpert-v1.0-small/train/patient00001/study1/...
1    CheXpert-v1.0-small/train/patient00002/study2/...
2    CheXpert-v1.0-small/train/patient00002/study1/...
3    CheXpert-v1.0-small/train/patient00002/study1/...
4    CheXpert-v1.0-small/train/patient00003/study1/...
Name: Path, dtype: str

In [13]:
val_df["Path"].head()

0    CheXpert-v1.0-small/valid/patient64541/study1/...
1    CheXpert-v1.0-small/valid/patient64542/study1/...
2    CheXpert-v1.0-small/valid/patient64542/study1/...
3    CheXpert-v1.0-small/valid/patient64543/study1/...
4    CheXpert-v1.0-small/valid/patient64544/study1/...
Name: Path, dtype: str

In [14]:
test_df["Study"].head()

0    CheXpert-v1.0/test/patient64741/study1
1    CheXpert-v1.0/test/patient64742/study1
2    CheXpert-v1.0/test/patient64743/study1
3    CheXpert-v1.0/test/patient64744/study1
4    CheXpert-v1.0/test/patient64745/study1
Name: Study, dtype: str

In [15]:
project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))

train_df["Path"] = train_df["Path"].str.replace(
    "CheXpert-v1.0-small/", "", n=1
)

In [16]:
for p in train_df["Path"].head():
    full = os.path.join(project_root, "data", "raw", p)
    print(full, "—", "EXISTS" if os.path.exists(full) else "MISSING")

d:\ATU\Dissertation\noisy-labels-medical-imaging\data\raw\train/patient00001/study1/view1_frontal.jpg — EXISTS
d:\ATU\Dissertation\noisy-labels-medical-imaging\data\raw\train/patient00002/study2/view1_frontal.jpg — EXISTS
d:\ATU\Dissertation\noisy-labels-medical-imaging\data\raw\train/patient00002/study1/view1_frontal.jpg — EXISTS
d:\ATU\Dissertation\noisy-labels-medical-imaging\data\raw\train/patient00002/study1/view2_lateral.jpg — EXISTS
d:\ATU\Dissertation\noisy-labels-medical-imaging\data\raw\train/patient00003/study1/view1_frontal.jpg — EXISTS


In [17]:
val_df["Path"] = val_df["Path"].str.replace(
    "CheXpert-v1.0-small/", "", n=1
)

In [18]:
for p in val_df["Path"].head():
    full = os.path.join(project_root, "data", "raw", p)
    print(full, "—", "EXISTS" if os.path.exists(full) else "MISSING")

d:\ATU\Dissertation\noisy-labels-medical-imaging\data\raw\valid/patient64541/study1/view1_frontal.jpg — EXISTS
d:\ATU\Dissertation\noisy-labels-medical-imaging\data\raw\valid/patient64542/study1/view1_frontal.jpg — EXISTS
d:\ATU\Dissertation\noisy-labels-medical-imaging\data\raw\valid/patient64542/study1/view2_lateral.jpg — EXISTS
d:\ATU\Dissertation\noisy-labels-medical-imaging\data\raw\valid/patient64543/study1/view1_frontal.jpg — EXISTS
d:\ATU\Dissertation\noisy-labels-medical-imaging\data\raw\valid/patient64544/study1/view1_frontal.jpg — EXISTS


In [19]:
test_df["Study"] = test_df["Study"].str.replace(
    "CheXpert-v1.0/", "", n=1
)

In [20]:
for p in test_df["Study"].head():
    full = os.path.join(project_root, "data", "raw", p)
    print(full, "—", "EXISTS" if os.path.exists(full) else "MISSING")

d:\ATU\Dissertation\noisy-labels-medical-imaging\data\raw\test/patient64741/study1 — MISSING
d:\ATU\Dissertation\noisy-labels-medical-imaging\data\raw\test/patient64742/study1 — MISSING
d:\ATU\Dissertation\noisy-labels-medical-imaging\data\raw\test/patient64743/study1 — MISSING
d:\ATU\Dissertation\noisy-labels-medical-imaging\data\raw\test/patient64744/study1 — MISSING
d:\ATU\Dissertation\noisy-labels-medical-imaging\data\raw\test/patient64745/study1 — MISSING


## 5. Exploratory Data Analysis

The following analyses examine:

- label value counts across the dataset;
- image dimensions and pixel-value characteristics;
- patient overlap between the training and validation splits;
- frontal and lateral view distributions; and
- class distributions and pathology co-occurrence patterns.


In [21]:
train_df["Path"].head()

0    train/patient00001/study1/view1_frontal.jpg
1    train/patient00002/study2/view1_frontal.jpg
2    train/patient00002/study1/view1_frontal.jpg
3    train/patient00002/study1/view2_lateral.jpg
4    train/patient00003/study1/view1_frontal.jpg
Name: Path, dtype: str

In [22]:
test_df["Study"].head()

0    test/patient64741/study1
1    test/patient64742/study1
2    test/patient64743/study1
3    test/patient64744/study1
4    test/patient64745/study1
Name: Study, dtype: str

In [23]:
label_cols = train_df.columns[5:]
print("Label columns:", list(label_cols))

for col in label_cols:
    print(f"\n--- {col} ---")
    print(train_df[col].value_counts(dropna=False))

Label columns: ['No Finding', 'Enlarged Cardiomediastinum', 'Cardiomegaly', 'Lung Opacity', 'Lung Lesion', 'Edema', 'Consolidation', 'Pneumonia', 'Atelectasis', 'Pneumothorax', 'Pleural Effusion', 'Pleural Other', 'Fracture', 'Support Devices']

--- No Finding ---
No Finding
NaN    201033
1.0     22381
Name: count, dtype: int64

--- Enlarged Cardiomediastinum ---
Enlarged Cardiomediastinum
 NaN    178575
 0.0     21638
-1.0     12403
 1.0     10798
Name: count, dtype: int64

--- Cardiomegaly ---
Cardiomegaly
 NaN    177211
 1.0     27000
 0.0     11116
-1.0      8087
Name: count, dtype: int64

--- Lung Opacity ---
Lung Opacity
 NaN    105636
 1.0    105581
 0.0      6599
-1.0      5598
Name: count, dtype: int64

--- Lung Lesion ---
Lung Lesion
 NaN    211470
 1.0      9186
-1.0      1488
 0.0      1270
Name: count, dtype: int64

--- Edema ---
Edema
 NaN    137458
 1.0     52246
 0.0     20726
-1.0     12984
Name: count, dtype: int64

--- Consolidation ---
Consolidation
 NaN    152792
 

In [24]:
sample_paths = train_df["Path"].sample(100, random_state=42)
widths, heights, min_vals, max_vals = [], [], [], []

for p in sample_paths:
    img = Image.open(os.path.join(project_root, "data", "raw", p))
    w, h = img.size
    arr = np.array(img)
    widths.append(w)
    heights.append(h)
    min_vals.append(arr.min())
    max_vals.append(arr.max())

print(f"Width  — min: {min(widths)}, max: {max(widths)}, unique: {len(set(widths))}")
print(f"Height — min: {min(heights)}, max: {max(heights)}, unique: {len(set(heights))}")
print(f"Pixel min: {min(min_vals)}, Pixel max: {max(max_vals)}")
print(f"Mode (color): {img.mode}")

Width  — min: 320, max: 390, unique: 10
Height — min: 320, max: 452, unique: 9
Pixel min: 0, Pixel max: 255
Mode (color): L


In [25]:
train_patients = train_df["Path"].str.extract(r"(patient\d+)")[0].unique()
val_patients = val_df["Path"].str.extract(r"(patient\d+)")[0].unique()

overlap = set(train_patients) & set(val_patients)
print(f"Train patients: {len(train_patients)}")
print(f"Valid patients: {len(val_patients)}")
print(f"Overlap: {len(overlap)}")

Train patients: 64540
Valid patients: 200
Overlap: 0


In [26]:
train_df["View"] = train_df["Path"].apply(
    lambda p: "Frontal" if "frontal" in p.lower() else "Lateral"
)
print(train_df["View"].value_counts())

View
Frontal    191027
Lateral     32387
Name: count, dtype: int64
